# 20 Essential pandas Functions

Companion to the [complete article](https://todatabeyond.com/blog/20-pandas-functions-for-80-of-your-data-science-tasks). Run all cells in order with Python 3.11+ on CPU. No API keys or GPU. Setup installs pinned pandas 3.1.0, NumPy 2.3.3, certifi and Matplotlib 3.10.9. The original author baby-name CSV is fetched at an immutable commit with certificate-verified HTTPS and checked by SHA-256.

All 52 original examples are represented. The real comma-separated file has 11,345 rows and no missing values. The CSV-options example deliberately constructs a separate semicolon/preamble/N/A fixture; the missing-data section deliberately constructs a separate five-row numeric fixture with three missing cells. Neither replaces or mutates the full original DataFrame. Explicit numeric-only grouping avoids aggregating strings in modern pandas. Forward/backward fill use ffill/bfill; interpolation is a separate operation. Multi-key and differently named-key merges receive complete explicit schemas.

The bar and pie plots count name records, not total births. All three plots render during execution. Every CSV writes into a unique temporary directory, including nested paths; six export variants are retained separately and read back in the regression cell, which cleans up only that temporary directory. index=False controls index serialization, not row selection.

The original historical prose and screenshots remain in the article. Current Output panels come from this notebook run. Old fillna method/downcast descriptions and prose about default filling, index masks or query performance are not current API guarantees. See the website import review for the source discrepancy ledger. No output values, plots or execution counts are committed in this notebook.


In [ ]:
%pip install --quiet pandas==3.1.0 numpy==2.3.3 certifi==2025.10.5 matplotlib==3.10.9
from importlib.metadata import version
for package in ['pandas', 'numpy', 'certifi', 'matplotlib']:
    print(package, version(package))


In [ ]:
%matplotlib inline
import hashlib
import io
import ssl
import warnings
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlopen
import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET_URL = 'https://raw.githubusercontent.com/youssefHosni/Efficient-Python-for-Data-Scientists-Book/2507c7f72e3cb08ef349047a4ca052bcc7307337/Datasets/Popular_Baby_Names.csv'
DATASET_SHA256 = '5e41e47c4b8a01901ef9a5eab2e17df35c8b4e29fdee7a063a3045094f1109bb'
with urlopen(DATASET_URL, context=ssl.create_default_context(cafile=certifi.where()), timeout=60) as response:
    payload = response.read()
assert hashlib.sha256(payload).hexdigest() == DATASET_SHA256
workspace = TemporaryDirectory(prefix='pandas-essential-functions-')
WORK_DIR = Path(workspace.name)
(WORK_DIR / 'path' / 'to').mkdir(parents=True)
DATASET_PATH = WORK_DIR / 'Popular_Baby_Names.csv'
DATASET_PATH.write_bytes(payload)
warnings.filterwarnings('error', category=FutureWarning)
warnings.filterwarnings('error', category=pd.errors.ChainedAssignmentError)
print('Original CSV downloaded; HTTPS and SHA-256 verified.')
print('CSV examples use a unique temporary directory.')


## Example 1


In [ ]:
df = pd.read_csv(DATASET_PATH)
df_original = df.copy(deep=True)
print("Shape:", df.shape)
print(df.head().to_string(index=False))


## Example 2


In [ ]:
# Separate parser fixture: semicolons, five preamble lines and a missing value.
csv_demo = df.head(8).astype({'Count': 'float64'}).copy()
csv_demo.loc[0, 'Count'] = np.nan
options_path = WORK_DIR / 'csv-options.csv'
options_path.write_text('preamble\n' * 5 + csv_demo.to_csv(sep=';', na_rep='N/A'))
df_options = pd.read_csv(options_path, sep=';', header=0, index_col=0,
                         skiprows=5, na_values='N/A')
print(df_options.to_string())


## Example 3


In [ ]:
numeric_description = df.describe()
print(numeric_description.to_string())


## Example 4


In [ ]:
all_description = df.describe(include='all')
print(all_description.to_string())


## Example 5


In [ ]:
text_description = df.describe(exclude='number')
print(text_description.to_string())


## Example 6


In [ ]:
df.info()


## Example 7


In [ ]:
gender_counts = df['Gender'].value_counts()
ax = gender_counts.plot(kind='bar', title='Baby-name records by gender', figsize=(7, 4))
plt.tight_layout()
print(gender_counts.to_string())
plt.show()
plt.close('all')


## Example 8


In [ ]:
gender_counts = df['Gender'].value_counts()
ax = gender_counts.plot(kind='pie', title='Baby-name records by gender', figsize=(7, 4))
plt.tight_layout()
print(gender_counts.to_string())
plt.show()
plt.close('all')


## Example 9


In [ ]:
ax = df['Count'].plot(kind='box', title='Distribution of the Count column', figsize=(7, 4))
plt.tight_layout()
print(df['Count'].describe().to_string())
plt.show()
plt.close('all')


## Example 10


In [ ]:
# Select the first row
print(df.iloc[0])


## Example 11


In [ ]:
# Select the first two rows
print(df.iloc[:2])


## Example 12


In [ ]:
# Select the first column
print(df.iloc[:, 0])


## Example 13


In [ ]:
# Select the first two columns
print(df.iloc[:, :2])


## Example 14


In [ ]:
# Select the (1, 1) element
print(df.iloc[1, 1])


## Example 15


In [ ]:
# Select the column name 'Gender'
print(df.loc[:,'Gender'])


## Example 16


In [ ]:
# Select the columns named 'Year of Birth' and 'Gender'
print(df.loc[:, ['Year of Birth', 'Gender']])


## Example 17


In [ ]:
df_assigned = df.assign(count_plus_5=df['Count'] + 5)
print(df_assigned.head().to_string(index=False))


## Example 18


In [ ]:
df_query_count = df.query('Count > 30 and Rank < 20')
print('Rows:', len(df_query_count))
print(df_query_count.head().to_string(index=False))


## Example 19


In [ ]:
df_query_gender = df.query("Gender == 'MALE'")
print("Rows:", len(df_query_gender))
print(df_query_gender.head().to_string(index=False))


## Example 20


In [ ]:
df_sorted_count = df.sort_values(by='Count')
print(df_sorted_count.head().to_string(index=False))


## Example 21


In [ ]:
df_sorted_rank = df.sort_values(by='Rank', ascending=False)
print(df_sorted_rank.head().to_string(index=False))


## Example 22


In [ ]:
df_sorted_multi = df.sort_values(by=['Count', 'Rank'])
print(df_sorted_multi.head().to_string(index=False))


## Example 23


In [ ]:
df_sample_22 = df.sample(n=2, replace=False, random_state=1, axis=0)
print(df_sample_22.to_string(index=False))


## Example 24


In [ ]:
df_sample_23 = df.sample(n=3, replace=True, random_state=1, axis=0)
print(df_sample_23.to_string(index=False))


## Example 25


In [ ]:
df_sample_24 = df.sample(n=2, replace=False, random_state=1, axis=1)
print(df_sample_24.head().to_string(index=False))


## Example 26


In [ ]:
missing_mask = df.isnull()
print(missing_mask.head().to_string(index=False))
print("Original missing cells:", int(missing_mask.sum().sum()))
# A separate five-row numeric fixture exercises the missing-data operations.
missing_df = df[['Count', 'Rank']].head(5).astype(float).copy()
missing_df.loc[[1, 3], 'Count'] = np.nan
missing_df.loc[3, 'Rank'] = np.nan
print("Missing-data fixture:")
print(missing_df.to_string())


## Example 27


In [ ]:
filled_zero = missing_df.fillna(0)
filled_forward = missing_df.ffill()
filled_backward = missing_df.bfill()
filled_interpolated = missing_df.interpolate()
for label, frame in [('Zero', filled_zero), ('Forward', filled_forward),
                     ('Backward', filled_backward), ('Interpolated', filled_interpolated)]:
    print(label)
    print(frame.to_string())


## Example 28


In [ ]:
filled_inplace = missing_df.copy(deep=True)
filled_inplace.fillna(0, inplace=True)
print(filled_inplace.to_string())


## Example 29


In [ ]:
complete_rows = missing_df.dropna()
print(complete_rows.to_string())
print("Shape:", complete_rows.shape)


## Example 30


In [ ]:
complete_columns = missing_df.dropna(axis=1)
print(complete_columns.to_string())
print("Shape:", complete_columns.shape)


## Example 31


In [ ]:
threshold_rows = missing_df.dropna(thresh=2)
print(threshold_rows.to_string())
print("Shape:", threshold_rows.shape)


## Example 32


In [ ]:
drop_first = df.drop(0)
print(drop_first.head().to_string())
print("Shape:", drop_first.shape)


## Example 33


In [ ]:
drop_two = df.drop([0, 1])
print(drop_two.head().to_string())
print("Shape:", drop_two.shape)


## Example 34


In [ ]:
drop_columns = df.drop(['Count', 'Rank'], axis=1)
print(drop_columns.head().to_string())
print("Shape:", drop_columns.shape)


## Example 35


In [ ]:
pivot_one = pd.pivot_table(df, index='Ethnicity', values='Count', aggfunc='sum')
print(pivot_one.head().to_string())


## Example 36


In [ ]:
pivot_two = pd.pivot_table(df, index=['Ethnicity', 'Gender'], values='Count', aggfunc=['sum', 'count'])
print(pivot_two.head(20).to_string())


## Example 37


In [ ]:
grouped_means = df.groupby('Gender').mean(numeric_only=True)
print(grouped_means.to_string())


## Example 38


In [ ]:
grouped_sums = df.groupby(['Gender', 'Ethnicity']).sum(numeric_only=True)
print(grouped_sums.to_string())


## Example 39


In [ ]:
df_transposed = df.transpose()
print(df_transposed.head())


## Example 40


In [ ]:
df_transposed_t = df.T
print(df_transposed_t.head())


## Example 41


In [ ]:
# Create the first DataFrame
df1 = pd.DataFrame({'key': ['A', 'B', 'C', 'D'],
                   'value': [1, 2, 3, 4]})

# Create the second DataFrame
df2 = pd.DataFrame({'key': ['B', 'D', 'E', 'F'],
                   'value': [5, 6, 7, 8]})

# Merge the two DataFrames on the 'key' column
merged_df = df1.merge(df2, on='key')

# Print the merged DataFrame
print(merged_df)


## Example 42


In [ ]:
left_multi = pd.DataFrame({'key1': ['B', 'B', 'D'], 'key2': [1, 2, 1], 'left_value': [2, 3, 4]})
right_multi = pd.DataFrame({'key1': ['B', 'B', 'E'], 'key2': [1, 2, 1], 'right_value': [5, 6, 7]})
merged_multi = left_multi.merge(right_multi, on=['key1', 'key2'])
print(merged_multi.to_string(index=False))


## Example 43


In [ ]:
left_named = df1.rename(columns={'key': 'key1'})
right_named = df2.rename(columns={'key': 'key3'})
merged_named = left_named.merge(right_named, left_on='key1', right_on='key3')
print(merged_named.to_string(index=False))


## Example 44


In [ ]:
renamed_count = df.rename(columns={'Count': 'count'})
print(renamed_count.head().to_string())


## Example 45


In [ ]:
renamed_columns = df.rename(columns={'Count': 'count', 'Rank': 'rank'})
print(renamed_columns.head().to_string())


## Example 46


In [ ]:
renamed_index = df.rename(index={0: 'first', 1: 'second', 2: 'third'})
print(renamed_index.head().to_string())


## Example 47


In [ ]:
export_46 = WORK_DIR / 'data.csv'
df.to_csv(export_46)
print('File: data.csv')
print('\n'.join(export_46.read_text().splitlines()[:4]))


## Example 48


In [ ]:
export_47 = WORK_DIR / 'path/to/data.csv'
df.to_csv(export_47)
print('File: path/to/data.csv')
print('\n'.join(export_47.read_text().splitlines()[:4]))


## Example 49


In [ ]:
export_48 = WORK_DIR / 'path/to/data.tsv'
df.to_csv(export_48, sep='\t')
print('File: path/to/data.tsv')
print('\n'.join(export_48.read_text().splitlines()[:4]))


## Example 50


In [ ]:
export_49 = WORK_DIR / 'path/to/subset.csv'
df.to_csv(export_49, columns=['Rank', 'Count'])
print('File: path/to/subset.csv')
print('\n'.join(export_49.read_text().splitlines()[:4]))


## Example 51


In [ ]:
export_50 = WORK_DIR / 'path/to/no-index.csv'
df.to_csv(export_50, index=False)
print('File: path/to/no-index.csv')
print('\n'.join(export_50.read_text().splitlines()[:4]))


## Example 52


In [ ]:
export_51 = WORK_DIR / 'path/to/missing.csv'
missing_df.to_csv(export_51, na_rep='NULL')
print('File: path/to/missing.csv')
print('\n'.join(export_51.read_text().splitlines()[:4]))


## Regression checks


In [ ]:
from pandas.testing import assert_frame_equal, assert_series_equal

assert version('pandas') == '3.1.0'
assert df.shape == (11345, 6)
assert df.isna().sum().sum() == 0
assert_frame_equal(df, df_original)
assert_frame_equal(df_options, csv_demo)
assert_frame_equal(numeric_description, df.select_dtypes(include='number').describe())
assert all_description.columns.tolist() == df.columns.tolist()
assert text_description.columns.tolist() == ['Gender', 'Ethnicity', "Child's First Name"]
assert gender_counts.to_dict() == {'FEMALE': 5806, 'MALE': 5539}
assert_series_equal(df_assigned['count_plus_5'], (df['Count'] + 5).rename('count_plus_5'))
assert_frame_equal(df_query_count, df.loc[(df['Count'] > 30) & (df['Rank'] < 20)])
assert_frame_equal(df_query_gender, df.loc[df['Gender'] == 'MALE'])
assert df_sorted_count['Count'].is_monotonic_increasing
assert df_sorted_rank['Rank'].is_monotonic_decreasing
assert_frame_equal(df_sorted_multi, df.sort_values(['Count', 'Rank']))
for i, n, replace, axis in [(22, 2, False, 0), (23, 3, True, 0), (24, 2, False, 1)]:
    assert_frame_equal(globals()['df_sample_' + str(i)], df.sample(n=n, replace=replace, random_state=1, axis=axis))
assert missing_df.isna().sum().to_dict() == {'Count': 2, 'Rank': 1}
assert_frame_equal(filled_inplace, filled_zero)
for frame in [filled_zero, filled_forward, filled_backward, filled_interpolated]:
    assert frame.notna().all().all()
    np.testing.assert_array_equal(frame.to_numpy()[missing_df.notna()], missing_df.to_numpy()[missing_df.notna()])
assert filled_forward.loc[1, 'Count'] == missing_df.loc[0, 'Count']
assert filled_backward.loc[1, 'Count'] == missing_df.loc[2, 'Count']
assert filled_interpolated.loc[1, 'Count'] == (missing_df.loc[0, 'Count'] + missing_df.loc[2, 'Count']) / 2
assert complete_rows.index.tolist() == threshold_rows.index.tolist() == [0, 2, 4]
assert complete_columns.shape == (5, 0)
assert drop_first.shape == (11344, 6) and 0 not in drop_first.index
assert drop_two.shape == (11343, 6) and not {0, 1}.intersection(drop_two.index)
assert drop_columns.columns.tolist() == df.columns[:-2].tolist()
assert pivot_one['Count'].sum() == df['Count'].sum()
assert pivot_two[('sum', 'Count')].sum() == df['Count'].sum()
assert pivot_two[('count', 'Count')].sum() == len(df)
numeric_columns = df.select_dtypes(include='number').columns.tolist()
for gender, group in df.groupby('Gender'):
    np.testing.assert_allclose(grouped_means.loc[gender], group[numeric_columns].mean())
assert grouped_sums['Count'].sum() == df['Count'].sum()
assert_frame_equal(df_transposed, df_transposed_t)
assert_frame_equal(df_transposed.T.infer_objects(), df)
assert merged_df['key'].tolist() == ['B', 'D']
assert merged_multi[['key1', 'key2']].values.tolist() == [['B', 1], ['B', 2]]
assert merged_named['key1'].tolist() == merged_named['key3'].tolist() == ['B', 'D']
assert 'count' in renamed_count and 'Count' not in renamed_count
assert renamed_columns.columns[-2:].tolist() == ['count', 'rank']
assert renamed_index.index[:3].tolist() == ['first', 'second', 'third']
for path in [export_46, export_47]:
    assert_frame_equal(pd.read_csv(path, index_col=0), df)
assert_frame_equal(pd.read_csv(export_48, sep='\t', index_col=0), df)
assert_frame_equal(pd.read_csv(export_49, index_col=0), df[['Rank', 'Count']])
assert_frame_equal(pd.read_csv(export_50), df)
assert_frame_equal(pd.read_csv(export_51, index_col=0, na_values='NULL'), missing_df)
assert export_51.read_text().count('NULL') == 3
assert len(list(WORK_DIR.rglob('*.csv'))) == 7
assert len(list(WORK_DIR.rglob('*.tsv'))) == 1
assert_frame_equal(df, df_original)
workspace.cleanup()
print('All full-dataset parsing, statistics, selection, cleaning, grouping, reshape, merge and CSV round-trip checks passed.')
print('Original data:', df.shape, '; missing cells:', int(df.isna().sum().sum()))
print('Source examples:', 52, '; plots:', 3, '; CSV export variants:', 6)
